# Deploy NVIDIA NIM from AWS Marketplace

NVIDIA NIM, a component of NVIDIA AI Enterprise, provides optimized inference for state-of-the-art AI models. This notebook demonstrates how to deploy and use the **NVIDIA Cosmos Transfer 2.5 2B** model from AWS Marketplace on Amazon SageMaker.

In this example we show how to deploy the NVIDIA Cosmos Transfer 2.5 2B model from AWS Marketplace on Amazon SageMaker on a p5.48xlarge SageMaker Instance.

Cosmos Transfer 2.5 2B is a **diffusion-based video-to-video generation model** that transforms input videos using multimodal control signals. Given a text prompt, an input video, and optional control signals (edge, segmentation, visibility, depth), the model generates a transformed output video. Unlike chat-based LLM/VLM NIMs, this model:

- Uses the `/v1/infer` endpoint (not `/v1/chat/completions`)
- Accepts video + control signal inputs (Base64-encoded or URL)
- Returns a base64-encoded MP4 video (`b64_video`)
- Requires Hopper architecture GPUs (H100/H200) with 80GB+ VRAM

Please check out the [NVIDIA NIM for Cosmos](https://docs.nvidia.com/nim/cosmos/2.0.0/release-notes.html) for more information.

## Pre-requisites:
1. **Note**: This notebook contains elements which render correctly in Jupyter interface. Open this notebook from an Amazon SageMaker Notebook Instance or Amazon SageMaker Studio.
1. Ensure that IAM role used has **AmazonSageMakerFullAccess**
1. To deploy this ML model successfully, ensure that:
    1. Either your IAM role has these three permissions and you have authority to make AWS Marketplace subscriptions in the AWS account used: 
        1. **aws-marketplace:ViewSubscriptions**
        1. **aws-marketplace:Unsubscribe**
        1. **aws-marketplace:Subscribe**  
    2. or your AWS account has a subscription to one of the models listed above.

**Important Notes**

- This model requires NVIDIA GPUs with Hopper architecture or later. The GPUs must have sufficient memory—refer to the [Supported Models section](https://docs.nvidia.com/nim/cosmos/2.0.0/support-matrix.html#cosmos-transfer2-5-matrix) for specific memory requirements by model..
- Video generation takes **several minutes** per request depending on video length and resolution.
- The model does **not** support streaming. Use `invoke_endpoint` (not `invoke_endpoint_with_response_stream`).
- At least one control modality (`edge`, `seg`, `vis`, or `depth`) must be provided in each request.

## Subscribe to the model package
To subscribe to the model package:
1. Open the model package listing page
1. On the AWS Marketplace listing, click on the **Continue to subscribe** button.
1. On the **Subscribe to this software** page, review and click on **"Accept Offer"** if you and your organization agrees with EULA, pricing, and support terms. 
1. Once you click on **Continue to configuration button** and then choose a **region**, you will see a **Product Arn** displayed. This is the model package ARN that you need to specify while creating a deployable model. Copy the ARN corresponding to your region and specify the same in the following cell.

In [ ]:
import boto3, json, sagemaker, time, os, base64
from sagemaker import get_execution_role, ModelPackage
from botocore.config import Config

config = Config(read_timeout=3600)
sess = boto3.Session()
sm = sess.client("sagemaker")
sagemaker_session = sagemaker.Session(boto_session=sess)
role = get_execution_role()
client = boto3.client("sagemaker-runtime", config=config)
region = sess.region_name

In [ ]:
# replace the arn below with the model package arn you want to deploy
nim_package = "ENTER PACKAGE NAME"

# Mapping for Model Packages
model_package_map = {
    "us-east-1": f"arn:aws:sagemaker:us-east-1:865070037744:model-package/{nim_package}",
    "us-east-2": f"arn:aws:sagemaker:us-east-2:057799348421:model-package/{nim_package}",
    "us-west-1": f"arn:aws:sagemaker:us-west-1:382657785993:model-package/{nim_package}",
    "us-west-2": f"arn:aws:sagemaker:us-west-2:594846645681:model-package/{nim_package}",
    "ca-central-1": f"arn:aws:sagemaker:ca-central-1:470592106596:model-package/{nim_package}",
    "eu-central-1": f"arn:aws:sagemaker:eu-central-1:446921602837:model-package/{nim_package}",
    "eu-west-1": f"arn:aws:sagemaker:eu-west-1:985815980388:model-package/{nim_package}",
    "eu-west-2": f"arn:aws:sagemaker:eu-west-2:856760150666:model-package/{nim_package}",
    "eu-west-3": f"arn:aws:sagemaker:eu-west-3:843114510376:model-package/{nim_package}",
    "eu-north-1": f"arn:aws:sagemaker:eu-north-1:136758871317:model-package/{nim_package}",
    "ap-southeast-1": f"arn:aws:sagemaker:ap-southeast-1:192199979996:model-package/{nim_package}",
    "ap-southeast-2": f"arn:aws:sagemaker:ap-southeast-2:666831318237:model-package/{nim_package}",
    "ap-northeast-2": f"arn:aws:sagemaker:ap-northeast-2:745090734665:model-package/{nim_package}",
    "ap-northeast-1": f"arn:aws:sagemaker:ap-northeast-1:977537786026:model-package/{nim_package}",
    "ap-south-1": f"arn:aws:sagemaker:ap-south-1:077584701553:model-package/{nim_package}",
    "sa-east-1": f"arn:aws:sagemaker:sa-east-1:270155090741:model-package/{nim_package}",
}

region = boto3.Session().region_name
if region not in model_package_map.keys():
    raise Exception(f"Current boto3 session region {region} is not supported.")

model_package_arn = model_package_map[region]
model_package_arn

## Create the SageMaker Endpoint

We first define SageMaker model using the specified ModelPackageArn.

In [ ]:
sm_model_name = "nim-cosmos-transfer2-5-2b"

create_model_response = sm.create_model(
    ModelName=sm_model_name,
    PrimaryContainer={
        'ModelPackageName': model_package_arn
    },
    ExecutionRoleArn=role,
    EnableNetworkIsolation=True
)
print("Model Arn: " + create_model_response["ModelArn"])

Next we create endpoint configuration specifying instance type. Cosmos Transfer 2.5 2B requires Hopper or later GPUs.

In [ ]:
endpoint_config_name = sm_model_name

create_endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            'VariantName': 'AllTraffic',
            'ModelName': sm_model_name,
            'InitialInstanceCount': 1,
            'InstanceType': 'ml.p5.48xlarge',
            'InferenceAmiVersion': 'al2-ami-sagemaker-inference-gpu-2',
            'RoutingConfig': {'RoutingStrategy': 'LEAST_OUTSTANDING_REQUESTS'},
            'ModelDataDownloadTimeoutInSeconds': 3600,
            'ContainerStartupHealthCheckTimeoutInSeconds': 3600,
        }
    ]
)
print("Endpoint Config Arn: " + create_endpoint_config_response["EndpointConfigArn"])

Using the above endpoint configuration we create a new sagemaker endpoint and wait for the deployment to finish. The status will change to InService once the deployment is successful.

In [ ]:
endpoint_name = endpoint_config_name
create_endpoint_response = sm.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name
)

print("Endpoint Arn: " + create_endpoint_response["EndpointArn"])

In [ ]:
resp = sm.describe_endpoint(EndpointName=endpoint_name)
status = resp["EndpointStatus"]
print("Status: " + status)

while status == "Creating":
    time.sleep(60)
    resp = sm.describe_endpoint(EndpointName=endpoint_name)
    status = resp["EndpointStatus"]
    print("Status: " + status)

print("Arn: " + resp["EndpointArn"])
print("Status: " + status)

### Run Inference

Once we have the model deployed we can run video generation inference. Cosmos Transfer 2.5 2B uses the `/v1/infer` API (routed via `/invocations` on SageMaker).

**Request format:**
- `prompt` (string): Text description of the desired output
- `video` (string): Input video as Base64-encoded string or URL
- `resolution` (string, optional): `"480"` or `"1024"`
- `edge`, `seg`, `vis`, `depth` (objects, optional): Control signals with `control_weight` (float) and `control` (video as Base64 or URL)

**Response format:**
- `b64_video`: Base64-encoded output video (MP4, VP9 codec)
- `seed`: The random seed used

For full API documentation, see the [Cosmos Transfer 2.5 API Reference](https://docs.nvidia.com/nim/cosmos/2.0.0/api-reference.html).

#### Helper functions

In [ ]:
def encode_video_to_base64(file_path):
    """Encode a local video file to base64"""
    with open(file_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def save_b64_video(b64_string, output_path):
    """Decode and save a base64-encoded video to disk"""
    video_bytes = base64.b64decode(b64_string)
    with open(output_path, "wb") as f:
        f.write(video_bytes)
    print(f"Video saved to {output_path} ({len(video_bytes):,} bytes)")

#### Download sample test videos

We download sample input and control signal videos for testing. These are small, low-resolution samples from the Cosmos Transfer 2.5 repository.

In [ ]:
import urllib.request

sample_base_url = "https://raw.githubusercontent.com/abhinavg4/cosmos-transfer2.5/main/assets_nim/low"
sample_files = {
    "robot_input.mp4": f"{sample_base_url}/robot_input.mp4",
    "robot_edge.mp4": f"{sample_base_url}/edge/robot_edge.mp4",
    "robot_seg.mp4": f"{sample_base_url}/seg/robot_seg.mp4",
    "robot_vis.mp4": f"{sample_base_url}/vis/robot_vis.mp4",
    "robot_depth.mp4": f"{sample_base_url}/depth/robot_depth.mp4",
}

os.makedirs("sample_videos", exist_ok=True)
for filename, url in sample_files.items():
    filepath = os.path.join("sample_videos", filename)
    if not os.path.exists(filepath):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filepath)
    else:
        print(f"  {filename} already exists, skipping.")

print("\nAll sample videos ready.")

#### Inference with all control signals

This example uses all four control signals (edge, segmentation, visibility, depth) for maximum control over the output.

In [ ]:
video_b64 = encode_video_to_base64('sample_videos/robot_input.mp4')
edge_b64 = encode_video_to_base64('sample_videos/robot_edge.mp4')
seg_b64 = encode_video_to_base64('sample_videos/robot_seg.mp4')
vis_b64 = encode_video_to_base64('sample_videos/robot_vis.mp4')
depth_b64 = encode_video_to_base64('sample_videos/robot_depth.mp4')

payload = {
    "prompt": "Two robotic arms manipulate blue fabric on a yellow cushion in a neutral lab setting.",
    "video": video_b64,
    "resolution": "480",
    "edge": {
        "control_weight": 1.0,
        "control": edge_b64
    },
    "seg": {
        "control_weight": 1.0,
        "control": seg_b64
    },
    "vis": {
        "control_weight": 1.0,
        "control": vis_b64
    },
    "depth": {
        "control_weight": 1.0,
        "control": depth_b64
    }
}

print("Sending inference request to SageMaker endpoint...")
print("(Video generation may take several minutes)")

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    Body=json.dumps(payload),
    ContentType="application/json",
    Accept="application/json",
)

output = json.loads(response["Body"].read().decode("utf8"))
print(f"Response keys: {list(output.keys())}")
print(f"Seed used: {output.get('seed')}")

if 'b64_video' in output:
    save_b64_video(output['b64_video'], 'output_all_controls.mp4')
else:
    print("Error: No b64_video in response")
    print(json.dumps(output, indent=2))

#### Inference with single edge control

You don't need to provide all control signals. Here we use only edge control for a simpler request. When a control signal's `control` video is omitted, the model auto-extracts it from the input video.

In [ ]:
video_b64 = encode_video_to_base64('sample_videos/robot_input.mp4')
edge_b64 = encode_video_to_base64('sample_videos/robot_edge.mp4')

payload_edge_only = {
    "prompt": "Two robotic arms manipulate blue fabric on a yellow cushion in a neutral lab setting.",
    "video": video_b64,
    "resolution": "480",
    "edge": {
        "control_weight": 1.0,
        "control": edge_b64
    }
}

print("Sending inference request with edge control only...")
print("(Video generation may take several minutes)")

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    Body=json.dumps(payload_edge_only),
    ContentType="application/json",
    Accept="application/json",
)

output = json.loads(response["Body"].read().decode("utf8"))
print(f"Response keys: {list(output.keys())}")
print(f"Seed used: {output.get('seed')}")

if 'b64_video' in output:
    save_b64_video(output['b64_video'], 'output_edge_only.mp4')
else:
    print("Error: No b64_video in response")
    print(json.dumps(output, indent=2))

#### Inference with auto-extracted control

When you provide a control type with only `control_weight` but no `control` video, the model auto-extracts the control signal from the input video.

In [ ]:
video_b64 = encode_video_to_base64('sample_videos/robot_input.mp4')

payload_auto = {
    "prompt": "Two robotic arms manipulate blue fabric on a yellow cushion in a neutral lab setting.",
    "video": video_b64,
    "resolution": "480",
    "edge": {
        "control_weight": 1.0
    }
}

print("Sending inference request with auto-extracted edge control...")
print("(Video generation may take several minutes)")

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    Body=json.dumps(payload_auto),
    ContentType="application/json",
    Accept="application/json",
)

output = json.loads(response["Body"].read().decode("utf8"))
print(f"Response keys: {list(output.keys())}")
print(f"Seed used: {output.get('seed')}")

if 'b64_video' in output:
    save_b64_video(output['b64_video'], 'output_auto_control.mp4')
else:
    print("Error: No b64_video in response")
    print(json.dumps(output, indent=2))

### Terminate endpoint and clean up artifacts

In [ ]:
sm.delete_model(ModelName=sm_model_name)
sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
sm.delete_endpoint(EndpointName=endpoint_name)